# Práctica Semana 2 — Transformación de Datos (ETL)
## Escuela Internacional de Gerencia — UIDE


## Descripción del Proyecto

El proyecto del grupo aborda el análisis de **ciberseguridad a nivel global**, cruzando tres fuentes de datos complementarias:

1. **`cybersecurity_threats`** — Base de datos PostgreSQL con registros de incidentes de ciberseguridad: tipo de ataque, industria objetivo, pérdida financiera y tiempo de resolución.
2. **`Cyber_security.csv`** — Índices globales de ciberseguridad por país: GCI, NCSI, CEI y DDL.
3. **`internet_users_by_country_cleaned.csv`** — Estadísticas de usuarios de internet por país: número absoluto, población y porcentaje de penetración.

El objetivo de esta práctica es aplicar la etapa **T (Transform)** del proceso ETL sobre estas tres fuentes, dejando los datos limpios, normalizados y listos para análisis o carga en un sistema de destino.

---

---
## SECCIÓN 1: Importación de Librerías

Se importan todas las librerías necesarias para el proceso ETL:

- **`pandas`**: manipulación y análisis de datos en estructuras tipo DataFrame.
- **`numpy`**: operaciones numéricas y manejo de valores nulos (`np.nan`).
- **`dotenv / load_dotenv`**: carga de variables de entorno desde el archivo `.env`.
- **`os`**: acceso a variables del sistema operativo.
- **`sqlalchemy / create_engine`**: conexión a la base de datos PostgreSQL.

In [1]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine

# Cargar variables de entorno desde el archivo .env
load_dotenv()

True

---
## SECCIÓN 2: Configuración Segura del Proyecto

### ¿Qué variables de entorno se utilizan?

Las siguientes variables se definen en el archivo `.env` (nunca se escriben directamente en el código):

```
POSTGRES_USER=nombre_usuario
POSTGRES_PASSWORD=contraseña_secreta
POSTGRES_DB=nombre_base_de_datos
POSTGRES_HOST=localhost
DATA_PATH=data/
```

### ¿Por qué NO se escriben credenciales en el código?

Escribir contraseñas directamente en el código es un **riesgo de seguridad crítico** porque:
- El archivo `.ipynb` puede subirse accidentalmente a repositorios públicos (GitHub, GitLab).
- Cualquier persona con acceso al notebook tendría acceso a la base de datos.
- Viola estándares de seguridad como ISO 27001 y PCI-DSS, relevantes en entornos financieros.
- Si las credenciales cambian, se deben modificar en múltiples archivos.

### ¿Cómo aporta a la seguridad del proyecto?

El archivo `.env` se agrega al `.gitignore` para que nunca sea versionado. Esto implementa el principio de **separación de configuración y código**, garantizando que las credenciales solo existan en la máquina local del desarrollador.

In [5]:

DB_USER     = os.getenv('POSTGRES_USER')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD')
DB_NAME     = os.getenv('POSTGRES_DB')
DB_HOST     = os.getenv('POSTGRES_HOST', 'localhost')
DATA_PATH   = os.getenv('DATA_PATH', 'data/')

### 2.1 — Carga de los tres datasets

Se cargan las tres fuentes de datos. La fuente de PostgreSQL requiere servidor activo;
en caso de no estar disponible se indica el error claramente sin detener la ejecución.
Los dos CSV se cargan directamente desde la ruta configurada en `DATA_PATH`.

In [9]:
# ── Fuente 1: Base de datos PostgreSQL ──────────────────────────────────────
# Se usa try/except para manejar el caso en que el servidor no esté activo
try:
    engine = create_engine(
        f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}'
    )
    df_threats = pd.read_sql('SELECT * FROM cybersecurity_threats', engine)
    print(f'✅ df_threats cargado desde PostgreSQL: {df_threats.shape}')
except Exception as e:
    print(f'⚠️  No se pudo conectar a PostgreSQL: {e}')

# ── Fuente 2: CSV Índices de Ciberseguridad ─────────────────────────────────
df_cib_eda = pd.read_csv(f'{DATA_PATH}Cyber_security.csv')
print(f'✅ df_cib_eda cargado: {df_cib_eda.shape}')

# ── Fuente 3: CSV Usuarios de Internet ──────────────────────────────────────
df_usenet = pd.read_csv(f'{DATA_PATH}internet_users_by_country_cleaned.csv')
print(f'✅ df_usenet cargado:  {df_usenet.shape}')

✅ df_threats cargado desde PostgreSQL: (3000, 11)
✅ df_cib_eda cargado: (192, 6)
✅ df_usenet cargado:  (215, 7)


---
## SECCIÓN 1: Exploración Inicial de las Fuentes de Datos

Antes de aplicar cualquier transformación se realiza una exploración sistemática
de las tres fuentes para identificar:
- Estructura general (filas, columnas, tipos de datos)
- Valores nulos o inconsistentes
- Registros duplicados
- Posibles columnas de relación entre fuentes
- Problemas de formato, escritura o codificación

### 1.1 — Exploración de `df_threats` (Amenazas de Ciberseguridad)

Se inspeccionan dimensiones, tipos de datos, nulos y duplicados del dataset
de incidentes de ciberseguridad extraído desde PostgreSQL.

In [11]:

print('FUENTE 1 — cybersecurity_threats')
# Dimensiones del dataset
print(f'Dimensiones : {df_threats.shape}')

# Registros completamente duplicados
print(f'Duplicados  : {df_threats.duplicated().sum()}')
print()

# Tipo de dato de cada columna
print('── Tipos de datos ──')
print(df_threats.dtypes)
print()

# Conteo de valores nulos por columna
print('── Valores nulos ──')
print(df_threats.isnull().sum())
print()

# Vista previa de las primeras filas
print('── Vista previa ──')
df_threats.head()

FUENTE 1 — cybersecurity_threats
Dimensiones : (3000, 11)
Duplicados  : 0

── Tipos de datos ──
id                                     int64
country                                  str
year                                   int64
attack_type                              str
target_industry                          str
financial_loss_in_million_usd        float64
number_of_affected_users               int64
attack_source                            str
security_vulnerability_type              str
defense_mechanism_used                   str
incident_resolution_time_in_hours      int64
dtype: object

── Valores nulos ──
id                                   0
country                              0
year                                 0
attack_type                          0
target_industry                      0
financial_loss_in_million_usd        0
number_of_affected_users             0
attack_source                        0
security_vulnerability_type          0
defense_mechanism_used 

,id,country,year,attack_type,target_industry,financial_loss_in_million_usd,number_of_affected_users,attack_source,security_vulnerability_type,defense_mechanism_used,incident_resolution_time_in_hours
0,1,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63
1,2,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71
2,3,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20
3,4,UK,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7
4,5,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68


**Hallazgos — `df_threats`:**
- Contiene registros de incidentes de ciberseguridad con variables financieras y operativas.
- Las columnas `Attack Type`, `Target Industry` y `Severity Level` son categóricas de texto y pueden presentar inconsistencias de capitalización (ej: `finance` vs `Finance`).
- La columna `Country` es la **clave de relación** con los otros dos datasets.
- Se deben verificar valores negativos en la columna de pérdida financiera, ya que serían datos inválidos.
- Los nombres de columna contienen espacios y paréntesis que deben normalizarse.

### 1.2 — Exploración de `df_cib_eda` (Índices de Ciberseguridad)

Se analiza el dataset de índices globales de ciberseguridad por país,
incluyendo el porcentaje de datos faltantes por columna.

In [12]:
print('═' * 60)
print('FUENTE 2 — Cyber_security.csv')
print('═' * 60)

# Dimensiones del dataset
print(f'Dimensiones : {df_cib_eda.shape}')

# Registros completamente duplicados
print(f'Duplicados  : {df_cib_eda.duplicated().sum()}')
print()

# Tipo de dato de cada columna
print('── Tipos de datos ──')
print(df_cib_eda.dtypes)
print()

# Nulos absolutos y porcentaje respecto al total de filas
print('── Valores nulos por columna ──')
nulos = df_cib_eda.isnull().sum()
pct   = (nulos / len(df_cib_eda) * 100).round(1)
print(pd.DataFrame({'Nulos': nulos, '% Faltante': pct}))
print()

# Categorías únicas de la columna Region
print('── Categorías únicas de Region ──')
print(df_cib_eda['Region'].unique())
print()

# Vista previa de las primeras filas
print('── Vista previa ──')
df_cib_eda.head()

════════════════════════════════════════════════════════════
FUENTE 2 — Cyber_security.csv
════════════════════════════════════════════════════════════
Dimensiones : (192, 6)
Duplicados  : 0

── Tipos de datos ──
Country        str
Region         str
CEI        float64
GCI        float64
NCSI       float64
DDL        float64
dtype: object

── Valores nulos por columna ──
         Nulos  % Faltante
Country      0         0.0
Region       0         0.0
CEI         84        43.8
GCI          2         1.0
NCSI        25        13.0
DDL         40        20.8

── Categorías únicas de Region ──
<StringArray>
['Asia-Pasific', 'Europe', 'Africa', 'North America', 'South America']
Length: 5, dtype: str

── Vista previa ──


,Country,Region,CEI,GCI,NCSI,DDL
0,Afghanistan,Asia-Pasific,1.000,5.20,11.69,19.50
1,Albania,Europe,0.566,64.32,62.34,48.74
2,Algeria,Africa,0.721,33.95,33.77,42.81
3,Andorra,Europe,NaN,26.38,NaN,NaN
4,Angola,Africa,NaN,12.99,9.09,22.69


**Hallazgos — `df_cib_eda`:**
- 192 países, 6 columnas. `Country` y `Region` están completas; las 4 columnas numéricas tienen nulos.
- **CEI tiene 84 nulos (44%)** — el porcentaje más alto; requiere imputación por grupo regional.
- GCI solo tiene 2 nulos — fácilmente imputables con la mediana regional.
- La columna `Region` contiene el error ortográfico **`Asia-Pasific`** en lugar de `Asia-Pacific`.
- La columna `Country` es la **clave de relación** con los otros datasets, pero puede tener diferencias de escritura.

### 1.3 — Exploración de `df_usenet` (Usuarios de Internet)

Se analiza el dataset de penetración de internet por país.
Este dataset fue previamente limpiado (`_cleaned`), lo que se verificará.

In [13]:

print('FUENTE 3 — internet_users_by_country_cleaned.csv')


# Dimensiones del dataset
print(f'Dimensiones : {df_usenet.shape}')

# Registros completamente duplicados
print(f'Duplicados  : {df_usenet.duplicated().sum()}')
print()

# Tipo de dato de cada columna
print('── Tipos de datos ──')
print(df_usenet.dtypes)
print()

# Conteo de valores nulos por columna
print('── Valores nulos ──')
print(df_usenet.isnull().sum())
print()

# Rango del año de los datos (los registros son de años distintos)
print('── Rango de años en la columna Year ──')
print(df_usenet['Year'].value_counts().sort_index())
print()

# Vista previa de las primeras filas
print('── Vista previa ──')
df_usenet.head()

FUENTE 3 — internet_users_by_country_cleaned.csv
Dimensiones : (215, 7)
Duplicados  : 0

── Tipos de datos ──
Country or area                  str
Subregion                        str
Region                           str
Internet users                 int64
Population_2021                int64
Year                           int64
Percentage_Internet_Users    float64
dtype: object

── Valores nulos ──
Country or area              0
Subregion                    0
Region                       0
Internet users               0
Population_2021              0
Year                         0
Percentage_Internet_Users    0
dtype: int64

── Rango de años en la columna Year ──
Year
2020      2
2021    184
2022      9
2023     18
2024      2
Name: count, dtype: int64

── Vista previa ──


,Country or area,Subregion,Region,Internet users,Population_2021,Year,Percentage_Internet_Users
0,China,Eastern Asia,Asia,1102140000,1425893465,2022,77.3
1,India,Southern Asia,Asia,881250000,1407563842,2024,62.6
2,United States,Northern America,Americas,311300000,336997624,2023,92.4
3,Indonesia,South-eastern Asia,Asia,215626156,273753191,2023,78.8
4,Pakistan,Southern Asia,Asia,170000000,240000000,2022,70.8


**Hallazgos — `df_usenet`:**
- 215 países/territorios, 7 columnas. Dataset **completamente limpio** (0 nulos en todas las columnas).
- La columna de país se llama `Country or area` — diferente a `Country` en los otros datasets. Debe renombrarse antes de cualquier join.
- La columna `Year` varía entre 2020–2024: los datos no son del mismo año para todos los países.
- `Internet users` y `Population_2021` son enteros de gran magnitud — candidatos para derivar un ratio por habitante.

### 1.4 — Tabla resumen: columnas de relación entre fuentes

| Fuente | Columna clave | Tipo | Observación |
|---|---|---|---|
| `df_threats` | `Country` | str | Nombre del país del incidente |
| `df_cib_eda` | `Country` | str | Nombre del país (mismo nombre de columna) |
| `df_usenet` | `Country or area` | str | Debe renombrarse a `country` antes del join |

Antes de unir las fuentes, se deben normalizar los nombres de país: misma capitalización, sin espacios extras, sin tildes o abreviaciones distintas.

---

## SECCIÓN 3: Limpieza de Datos

Se definen funciones reutilizables para aplicar la limpieza de forma organizada,
comentada y reproducible sobre cada dataset. Cada función tiene una responsabilidad
única y puede aplicarse a cualquiera de las tres fuentes.

### 3.1 — Función: normalizar nombres de columnas

Convierte todos los nombres de columnas a un formato estándar:
minúsculas, sin espacios (reemplazados por `_`) y sin caracteres especiales.

In [15]:
def limpiar_nombres_columnas(df):

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace(r'[()$]', '', regex=True)
        .str.replace(r'_+', '_', regex=True)
        .str.strip('_')
    )
    return df

### 3.2 — Función: eliminar registros duplicados

Detecta y elimina filas completamente duplicadas. Imprime un reporte
con la cantidad de registros eliminados para trazabilidad.

In [16]:
def eliminar_duplicados(df, nombre_df):

    antes   = len(df)
    df      = df.drop_duplicates()
    despues = len(df)
    eliminados = antes - despues
    print(f'[{nombre_df}] Duplicados eliminados: {eliminados} '
          f'(antes: {antes} filas → después: {despues} filas)')
    return df

### 3.3 — Función: normalizar texto en columnas categóricas

Estandariza el formato de columnas de texto para eliminar inconsistencias
de capitalización y espacios que impiden comparaciones correctas.

In [17]:
def normalizar_texto_columna(df, columna):
    df[columna] = df[columna].astype(str).str.strip().str.title()
    return df

### 3.4 — Función: imputar nulos con mediana por grupo

Imputa valores faltantes usando la mediana calculada dentro de cada grupo
(ej: mediana del GCI para los países de la misma región). Si algún grupo
no tiene ningún valor válido, se aplica la mediana global como respaldo.

In [18]:
def imputar_mediana_por_grupo(df, columna_valor, columna_grupo):

    # Calcular mediana global antes de modificar el DataFrame
    mediana_global = df[columna_valor].median()

    # Imputar con mediana del grupo
    df[columna_valor] = df.groupby(columna_grupo)[columna_valor].transform(
        lambda x: x.fillna(x.median())
    )

    # Respaldo: imputar con mediana global si quedaron nulos
    nulos_restantes = df[columna_valor].isnull().sum()
    if nulos_restantes > 0:
        df[columna_valor] = df[columna_valor].fillna(mediana_global)
        print(f'  ↳ {nulos_restantes} nulos residuales imputados '
              f'con mediana global ({mediana_global:.2f})')

    return df

### 3.5 — Aplicar limpieza a `df_threats`

Se aplican en orden:
1. Normalización de nombres de columnas.
2. Eliminación de duplicados exactos.
3. Normalización de texto en columnas categóricas.
4. Eliminación de registros con pérdida financiera negativa (datos inválidos).
5. Imputación de nulos numéricos con la mediana global.

In [19]:
print('── Limpieza df_threats ──')

df_threats = limpiar_nombres_columnas(df_threats)
print('Columnas normalizadas:', df_threats.columns.tolist())

# Paso 2: Eliminar filas completamente duplicadas
df_threats = eliminar_duplicados(df_threats, 'df_threats')

# Paso 3: Normalizar columnas de texto categórico
# Esto unifica variantes como 'finance', 'Finance', 'FINANCE' → 'Finance'
for col in ['attack_type', 'target_industry', 'severity_level', 'country']:
    if col in df_threats.columns:
        df_threats = normalizar_texto_columna(df_threats, col)
        print(f'Texto normalizado en columna: {col}')

# Paso 4: Eliminar registros con pérdida financiera negativa
# Una pérdida negativa no tiene sentido en el contexto del dataset
col_perdida = 'financial_loss_in_million'
if col_perdida in df_threats.columns:
    invalidos = (df_threats[col_perdida] < 0).sum()
    df_threats = df_threats[df_threats[col_perdida] >= 0]
    print(f'Registros eliminados (pérdida < 0): {invalidos}')

# Paso 5: Imputar nulos numéricos con la mediana global de cada columna
for col_num in ['financial_loss_in_million', 'incident_resolution_time_in_hours']:
    if col_num in df_threats.columns:
        nulos_col = df_threats[col_num].isnull().sum()
        if nulos_col > 0:
            mediana = df_threats[col_num].median()
            df_threats[col_num] = df_threats[col_num].fillna(mediana)
            print(f'Nulos en "{col_num}" imputados con mediana: {mediana:.2f}')

print(f'\n✅ df_threats limpio — Dimensiones finales: {df_threats.shape}')
print(f'   Nulos restantes: {df_threats.isnull().sum().sum()}')

── Limpieza df_threats ──
Columnas normalizadas: ['id', 'country', 'year', 'attack_type', 'target_industry', 'financial_loss_in_million_usd', 'number_of_affected_users', 'attack_source', 'security_vulnerability_type', 'defense_mechanism_used', 'incident_resolution_time_in_hours']
[df_threats] Duplicados eliminados: 0 (antes: 3000 filas → después: 3000 filas)
Texto normalizado en columna: attack_type
Texto normalizado en columna: target_industry
Texto normalizado en columna: country

✅ df_threats limpio — Dimensiones finales: (3000, 11)
   Nulos restantes: 0


### 3.6 — Aplicar limpieza a `df_cib_eda`

Se aplican en orden:
1. Normalización de nombres de columnas.
2. Eliminación de duplicados.
3. Corrección del error ortográfico `Asia-Pasific` → `Asia-Pacific`.
4. Normalización de texto en columnas categóricas.
5. Imputación de nulos numéricos con mediana regional (más representativa que la global).

In [21]:
print('── Limpieza df_cib_eda ──')

# Paso 1: Normalizar nombres de columnas
df_cib_eda = limpiar_nombres_columnas(df_cib_eda)
print('Columnas normalizadas:', df_cib_eda.columns.tolist())

# Paso 2: Eliminar duplicados exactos
df_cib_eda = eliminar_duplicados(df_cib_eda, 'df_cib_eda')

# Paso 3: Corregir error ortográfico en la columna 'region'
# 'Asia-Pasific' es un error de escritura que debe corregirse
# antes de usar esta columna como agrupador de imputación
df_cib_eda['region'] = df_cib_eda['region'].str.replace(
    'Asia-Pasific', 'Asia-Pacific', regex=False
)
print('Categorías de región después de corrección:', df_cib_eda['region'].unique())

# Paso 4: Normalizar texto en columnas categóricas
df_cib_eda = normalizar_texto_columna(df_cib_eda, 'country')
df_cib_eda = normalizar_texto_columna(df_cib_eda, 'region')

# Paso 5: Imputar nulos numéricos con mediana por región
# Se usa mediana regional porque los índices de ciberseguridad
# varían significativamente entre regiones geográficas
df_cib_eda = imputar_mediana_por_grupo(df_cib_eda, 'gci', 'region')
df_cib_eda = imputar_mediana_por_grupo(df_cib_eda, 'ncsi', 'region')
df_cib_eda = imputar_mediana_por_grupo(df_cib_eda, 'ddl', 'region')
df_cib_eda = imputar_mediana_por_grupo(df_cib_eda, 'cei', 'region')

print(f'\n✅ df_cib_eda limpio — Dimensiones finales: {df_cib_eda.shape}')
print(f'   Nulos restantes: {df_cib_eda.isnull().sum().sum()}')

── Limpieza df_cib_eda ──
Columnas normalizadas: ['country', 'region', 'cei', 'gci', 'ncsi', 'ddl']
[df_cib_eda] Duplicados eliminados: 0 (antes: 192 filas → después: 192 filas)
Categorías de región después de corrección: <StringArray>
['Asia-Pacific', 'Europe', 'Africa', 'North America', 'South America']
Length: 5, dtype: str

✅ df_cib_eda limpio — Dimensiones finales: (192, 6)
   Nulos restantes: 0


### 3.7 — Aplicar limpieza a `df_usenet`

Se aplican en orden:
1. Normalización de nombres de columnas.
2. Renombrar `country_or_area` → `country` para unificar la clave de relación.
3. Eliminación de duplicados.
4. Normalización de texto en columnas categóricas.
5. Verificación de coherencia: el porcentaje de usuarios debe estar entre 0 y 100.

In [22]:
print('── Limpieza df_usenet ──')

# Paso 1: Normalizar nombres de columnas
# 'Country or area' → 'country_or_area', 'Internet users' → 'internet_users', etc.
df_usenet = limpiar_nombres_columnas(df_usenet)
print('Columnas normalizadas:', df_usenet.columns.tolist())

# Paso 2: Renombrar columna clave para unificar con los otros datasets
# Los tres datasets deben compartir el mismo nombre de columna ('country')
# para poder realizar joins entre ellos
df_usenet.rename(columns={'country_or_area': 'country'}, inplace=True)
print('Columna "country_or_area" renombrada a "country"')

# Paso 3: Eliminar duplicados exactos
df_usenet = eliminar_duplicados(df_usenet, 'df_usenet')

# Paso 4: Normalizar texto en columnas categóricas
df_usenet = normalizar_texto_columna(df_usenet, 'country')
df_usenet = normalizar_texto_columna(df_usenet, 'subregion')
df_usenet = normalizar_texto_columna(df_usenet, 'region')

# Paso 5: Verificar coherencia del porcentaje de usuarios
# El porcentaje de usuarios de internet solo puede estar entre 0 y 100
invalidos_pct = df_usenet[
    (df_usenet['percentage_internet_users'] < 0) |
    (df_usenet['percentage_internet_users'] > 100)
]
print(f'Registros con porcentaje fuera de rango [0-100]: {len(invalidos_pct)}')

print(f'\n✅ df_usenet limpio — Dimensiones finales: {df_usenet.shape}')
print(f'   Nulos restantes: {df_usenet.isnull().sum().sum()}')

── Limpieza df_usenet ──
Columnas normalizadas: ['country_or_area', 'subregion', 'region', 'internet_users', 'population_2021', 'year', 'percentage_internet_users']
Columna "country_or_area" renombrada a "country"
[df_usenet] Duplicados eliminados: 0 (antes: 215 filas → después: 215 filas)
Registros con porcentaje fuera de rango [0-100]: 0

✅ df_usenet limpio — Dimensiones finales: (215, 7)
   Nulos restantes: 0


---
## SECCIÓN 4: Selección de Columnas Relevantes

Se evalúan las columnas de cada dataset según su **relevancia para el objetivo del proyecto**:
analizar el impacto de los incidentes de ciberseguridad en relación con el nivel de madurez
en ciberseguridad y la penetración de internet por país.

### 4.1 — Selección en `df_threats`

| Columna | Acción | Justificación |
|---|---|---|
| `country` | ✅ Mantener | Clave de relación con los otros datasets |
| `attack_type` | ✅ Mantener | Clasifica el tipo de incidente |
| `target_industry` | ✅ Mantener | Permite análisis por sector |
| `financial_loss_in_million` | ✅ Mantener | Métrica de impacto económico |
| `incident_resolution_time_in_hours` | ✅ Mantener | Métrica de eficiencia de respuesta |
| `severity_level` | ✅ Mantener | Permite priorizar por criticidad |

In [23]:
# Definir las columnas que se conservan en df_threats
cols_threats = [
    'country',
    'attack_type',
    'target_industry',
    'financial_loss_in_million',
    'incident_resolution_time_in_hours',
    'severity_level',
]

# Se filtran solo las columnas que existen en el DataFrame
# (previene errores si el dataset de ejemplo no tiene todas)
cols_threats_ok = [c for c in cols_threats if c in df_threats.columns]
df_threats_sel  = df_threats[cols_threats_ok].copy()

print('Columnas seleccionadas:', df_threats_sel.columns.tolist())
print(f'Dimensiones resultantes: {df_threats_sel.shape}')
df_threats_sel.head()

Columnas seleccionadas: ['country', 'attack_type', 'target_industry', 'incident_resolution_time_in_hours']
Dimensiones resultantes: (3000, 4)


,country,attack_type,target_industry,incident_resolution_time_in_hours
0,China,Phishing,Education,63
1,China,Ransomware,Retail,71
2,India,Man-In-The-Middle,It,20
3,Uk,Ransomware,Telecommunications,7
4,Germany,Man-In-The-Middle,It,68


### 4.2 — Selección en `df_cib_eda`

| Columna | Acción | Justificación |
|---|---|---|
| `country` | ✅ Mantener | Clave de relación |
| `region` | ✅ Mantener | Permite análisis geográfico agrupado |
| `gci` | ✅ Mantener | Índice principal de madurez en ciberseguridad |
| `ncsi` | ✅ Mantener | Complementa GCI con capacidad nacional |
| `cei` | ✅ Mantener | Mide exposición a amenazas cibernéticas |
| `ddl` | ✅ Mantener | Nivel de desarrollo digital del país |

Todas las columnas se conservan: cada índice aporta una dimensión diferente del análisis.

In [24]:
# Definir las columnas que se conservan en df_cib_eda
cols_cib_eda    = ['country', 'region', 'gci', 'ncsi', 'cei', 'ddl']
cols_cib_eda_ok = [c for c in cols_cib_eda if c in df_cib_eda.columns]
df_cib_eda_sel  = df_cib_eda[cols_cib_eda_ok].copy()

print('Columnas seleccionadas:', df_cib_eda_sel.columns.tolist())
print(f'Dimensiones resultantes: {df_cib_eda_sel.shape}')
df_cib_eda_sel.head()

Columnas seleccionadas: ['country', 'region', 'gci', 'ncsi', 'cei', 'ddl']
Dimensiones resultantes: (192, 6)


,country,region,gci,ncsi,cei,ddl
0,Afghanistan,Asia-Pacific,5.20,11.69,1.000,19.50
1,Albania,Europe,64.32,62.34,0.566,48.74
2,Algeria,Africa,33.95,33.77,0.721,42.81
3,Andorra,Europe,26.38,75.32,0.286,68.46
4,Angola,Africa,12.99,9.09,0.707,22.69


### 4.3 — Selección en `df_usenet`

| Columna | Acción | Justificación |
|---|---|---|
| `country` | ✅ Mantener | Clave de relación |
| `region` | ✅ Mantener | Análisis regional |
| `internet_users` | ✅ Mantener | Volumen absoluto de usuarios |
| `percentage_internet_users` | ✅ Mantener | Métrica relativa comparable entre países |
| `population_2021` | ✅ Mantener | Base demográfica para cálculos de ratio |
| `subregion` | ❌ Eliminar | Demasiado granular; `region` es suficiente |
| `year` | ❌ Eliminar | Los registros son de años distintos; no es variable de análisis central |

In [25]:
# Definir las columnas que se conservan en df_usenet
# Se excluyen: 'subregion' (demasiado granular) y 'year' (datos de años distintos)
cols_usenet    = ['country', 'region', 'internet_users',
                  'percentage_internet_users', 'population_2021']
cols_usenet_ok = [c for c in cols_usenet if c in df_usenet.columns]
df_usenet_sel  = df_usenet[cols_usenet_ok].copy()

print('Columnas seleccionadas:', df_usenet_sel.columns.tolist())
print('Columnas eliminadas: subregion, year')
print(f'Dimensiones resultantes: {df_usenet_sel.shape}')
df_usenet_sel.head()

Columnas seleccionadas: ['country', 'region', 'internet_users', 'percentage_internet_users', 'population_2021']
Columnas eliminadas: subregion, year
Dimensiones resultantes: (215, 5)


,country,region,internet_users,percentage_internet_users,population_2021
0,China,Asia,1102140000,77.3,1425893465
1,India,Asia,881250000,62.6,1407563842
2,United States,Americas,311300000,92.4,336997624
3,Indonesia,Asia,215626156,78.8,273753191
4,Pakistan,Asia,170000000,70.8,240000000


---
## SECCIÓN 5: Transformaciones Requeridas

Se aplican transformaciones para enriquecer los datos con nuevas columnas derivadas,
categorías normalizadas y métricas calculadas. Cada transformación se explica antes
del bloque de código correspondiente.

### 5.1 — Transformaciones sobre `df_threats_sel`

**Transformación 1 — `nivel_impacto_financiero`**

Categoriza la pérdida financiera en tramos para facilitar la priorización
de incidentes. Se usa `pd.cut()` que divide un rango numérico continuo
en intervalos con etiquetas descriptivas.

In [26]:
# Transformación 1: Categorizar pérdida financiera en tramos
# bins define los límites de cada intervalo (en millones de dólares)
# labels define la etiqueta descriptiva de cada tramo

if 'financial_loss_in_million' in df_threats_sel.columns:
    df_threats_sel['nivel_impacto_financiero'] = pd.cut(
        df_threats_sel['financial_loss_in_million'],
        bins   = [0, 1, 5, 20, float('inf')],
        labels = ['Bajo (<1M)', 'Medio (1-5M)', 'Alto (5-20M)', 'Crítico (>20M)'],
        right  = True   # el límite derecho de cada intervalo es inclusivo
    )
    print('Distribución de niveles de impacto financiero:')
    print(df_threats_sel['nivel_impacto_financiero'].value_counts())

**Transformación 2 — `velocidad_respuesta`**

Clasifica el tiempo de resolución del incidente en tres categorías
usando una función `lambda`. Permite identificar qué incidentes
tardaron más de lo aceptable en ser contenidos.

In [27]:
# Transformación 2: Clasificar velocidad de respuesta con función lambda
# Se aplica .apply() con una lambda que evalúa el número de horas
# y retorna la categoría correspondiente

if 'incident_resolution_time_in_hours' in df_threats_sel.columns:
    df_threats_sel['velocidad_respuesta'] = df_threats_sel[
        'incident_resolution_time_in_hours'
    ].apply(
        lambda h: 'Rápida (<24h)'
                  if h < 24
                  else ('Normal (24-72h)'
                        if h <= 72
                        else 'Lenta (>72h)')
    )
    print('Distribución de velocidad de respuesta:')
    print(df_threats_sel['velocidad_respuesta'].value_counts())

df_threats_sel.head()

Distribución de velocidad de respuesta:
velocidad_respuesta
Normal (24-72h)    2042
Rápida (<24h)       958
Name: count, dtype: int64


,country,attack_type,target_industry,incident_resolution_time_in_hours,velocidad_respuesta
0,China,Phishing,Education,63,Normal (24-72h)
1,China,Ransomware,Retail,71,Normal (24-72h)
2,India,Man-In-The-Middle,It,20,Rápida (<24h)
3,Uk,Ransomware,Telecommunications,7,Rápida (<24h)
4,Germany,Man-In-The-Middle,It,68,Normal (24-72h)


### 5.2 — Transformaciones sobre `df_cib_eda_sel`

**Transformación 3 — `nivel_madurez_gci`**

Clasifica el GCI (0-100) en cuatro niveles de madurez en ciberseguridad
basados en estándares de categorización de la ITU.

In [28]:
# Transformación 3: Categorizar GCI en niveles de madurez
# Basado en los umbrales de clasificación del Global Cybersecurity Index (ITU)

if 'gci' in df_cib_eda_sel.columns:
    df_cib_eda_sel['nivel_madurez_gci'] = pd.cut(
        df_cib_eda_sel['gci'],
        bins   = [0, 25, 50, 75, 100],
        labels = [
            'Iniciando (0-25)',
            'En desarrollo (25-50)',
            'Avanzado (50-75)',
            'Líder (75-100)'
        ],
        right  = True
    )
    print('Distribución de niveles de madurez GCI:')
    print(df_cib_eda_sel['nivel_madurez_gci'].value_counts().sort_index())

Distribución de niveles de madurez GCI:
nivel_madurez_gci
Iniciando (0-25)         60
En desarrollo (25-50)    31
Avanzado (50-75)         26
Líder (75-100)           75
Name: count, dtype: int64


**Transformación 4 — `indice_compuesto`**

Crea un índice sintético ponderado que combina GCI (60%) y NCSI (40%).
El GCI tiene mayor peso porque es el estándar más reconocido internacionalmente.
Este índice permite un ranking único de ciberseguridad por país.

In [29]:
# Transformación 4: Índice compuesto ponderado
# GCI tiene peso del 60% por ser el estándar principal de la ITU
# NCSI tiene peso del 40% como complemento de capacidad nacional

if 'gci' in df_cib_eda_sel.columns and 'ncsi' in df_cib_eda_sel.columns:
    df_cib_eda_sel['indice_compuesto'] = (
        df_cib_eda_sel['gci']  * 0.6 +
        df_cib_eda_sel['ncsi'] * 0.4
    ).round(2)

    print('Estadísticas del índice compuesto:')
    print(df_cib_eda_sel['indice_compuesto'].describe())

df_cib_eda_sel.head()

Estadísticas del índice compuesto:
count    192.000000
mean      48.404219
std       29.666058
min        3.970000
25%       18.245000
50%       48.115000
75%       75.922500
max       97.090000
Name: indice_compuesto, dtype: float64


,country,region,gci,ncsi,cei,ddl,nivel_madurez_gci,indice_compuesto
0,Afghanistan,Asia-Pacific,5.20,11.69,1.000,19.50,Iniciando (0-25),7.80
1,Albania,Europe,64.32,62.34,0.566,48.74,Avanzado (50-75),63.53
2,Algeria,Africa,33.95,33.77,0.721,42.81,En desarrollo (25-50),33.88
3,Andorra,Europe,26.38,75.32,0.286,68.46,En desarrollo (25-50),45.96
4,Angola,Africa,12.99,9.09,0.707,22.69,Iniciando (0-25),11.43


### 5.3 — Transformaciones sobre `df_usenet_sel`

**Transformación 5 — `categoria_penetracion`**

Clasifica a los países en cuatro categorías según el porcentaje
de población que usa internet, útil para análisis de brecha digital.

In [30]:
# Transformación 5: Categorizar penetración de internet
# Los umbrales reflejan niveles de inclusión digital global

if 'percentage_internet_users' in df_usenet_sel.columns:
    df_usenet_sel['categoria_penetracion'] = pd.cut(
        df_usenet_sel['percentage_internet_users'],
        bins   = [0, 30, 60, 80, 100],
        labels = [
            'Baja (<30%)',
            'Media (30-60%)',
            'Alta (60-80%)',
            'Muy Alta (>80%)'
        ],
        right  = True
    )
    print('Distribución de categorías de penetración:')
    print(df_usenet_sel['categoria_penetracion'].value_counts().sort_index())

Distribución de categorías de penetración:
categoria_penetracion
Baja (<30%)        53
Media (30-60%)     49
Alta (60-80%)      53
Muy Alta (>80%)    60
Name: count, dtype: int64


**Transformación 6 — `usuarios_por_millon_hab`**

Calcula el número de usuarios de internet por cada millón de habitantes.
Esta métrica normaliza el volumen absoluto por el tamaño de la población,
haciendo comparables a países grandes y pequeños.

In [31]:
# Transformación 6: Ratio de usuarios por millón de habitantes
# Fórmula: (usuarios / población) × 1.000.000
# Se convierte a entero porque la fracción no aporta información relevante

if 'internet_users' in df_usenet_sel.columns and 'population_2021' in df_usenet_sel.columns:
    df_usenet_sel['usuarios_por_millon_hab'] = (
        df_usenet_sel['internet_users'] /
        df_usenet_sel['population_2021'] * 1_000_000
    ).round(0).astype(int)

    print('Estadísticas del ratio usuarios por millón de hab.:')
    print(df_usenet_sel['usuarios_por_millon_hab'].describe())

df_usenet_sel.head()

Estadísticas del ratio usuarios por millón de hab.:
count        215.000000
mean      573690.297674
std       292692.033570
min        17278.000000
25%       302210.000000
50%       641053.000000
75%       820450.000000
max      1000000.000000
Name: usuarios_por_millon_hab, dtype: float64


,country,region,internet_users,percentage_internet_users,population_2021,categoria_penetracion,usuarios_por_millon_hab
0,China,Asia,1102140000,77.3,1425893465,Alta (60-80%),772947
1,India,Asia,881250000,62.6,1407563842,Alta (60-80%),626082
2,United States,Americas,311300000,92.4,336997624,Muy Alta (>80%),923745
3,Indonesia,Asia,215626156,78.8,273753191,Alta (60-80%),787666
4,Pakistan,Asia,170000000,70.8,240000000,Alta (60-80%),708333


---
## SECCIÓN 6: Generación de Identificadores

Se crean identificadores únicos secuenciales para los datasets provenientes de CSV.
La tabla de PostgreSQL ya genera su propio ID en la base de datos.

| Identificador | Dataset | Para qué se usa |
|---|---|---|
| `id_pais_cib` | `df_cib_eda_sel` | Identificar de forma unívoca cada país en el dataset de índices de ciberseguridad |
| `id_pais_net` | `df_usenet_sel` | Identificar de forma unívoca cada país en el dataset de usuarios de internet |

Estos IDs facilitan:
- El manejo de registros en operaciones de merge o join.
- La trazabilidad de cada fila durante el proceso ETL.
- La carga futura en una base de datos relacional como clave primaria.

In [32]:
# Generar ID secuencial para df_cib_eda_sel
# reset_index(drop=True) garantiza que el índice empiece en 0 antes de crear el ID
# insert(0, ...) coloca el ID como primera columna del DataFrame
df_cib_eda_sel = df_cib_eda_sel.reset_index(drop=True)
df_cib_eda_sel.insert(0, 'id_pais_cib', range(1, len(df_cib_eda_sel) + 1))

print('IDs generados en df_cib_eda_sel:')
print(f'  Rango    : {df_cib_eda_sel["id_pais_cib"].min()} — '
      f'{df_cib_eda_sel["id_pais_cib"].max()}')
print(f'  Únicos   : {df_cib_eda_sel["id_pais_cib"].nunique()} '
      f'de {len(df_cib_eda_sel)} registros')
print(f'  Duplicados en ID: {df_cib_eda_sel["id_pais_cib"].duplicated().sum()}')

# Generar ID secuencial para df_usenet_sel
df_usenet_sel = df_usenet_sel.reset_index(drop=True)
df_usenet_sel.insert(0, 'id_pais_net', range(1, len(df_usenet_sel) + 1))

print('\nIDs generados en df_usenet_sel:')
print(f'  Rango    : {df_usenet_sel["id_pais_net"].min()} — '
      f'{df_usenet_sel["id_pais_net"].max()}')
print(f'  Únicos   : {df_usenet_sel["id_pais_net"].nunique()} '
      f'de {len(df_usenet_sel)} registros')
print(f'  Duplicados en ID: {df_usenet_sel["id_pais_net"].duplicated().sum()}')

print('\n✅ Identificadores generados correctamente')
df_cib_eda_sel[['id_pais_cib', 'country', 'region', 'gci', 'nivel_madurez_gci']].head()

IDs generados en df_cib_eda_sel:
  Rango    : 1 — 192
  Únicos   : 192 de 192 registros
  Duplicados en ID: 0

IDs generados en df_usenet_sel:
  Rango    : 1 — 215
  Únicos   : 215 de 215 registros
  Duplicados en ID: 0

✅ Identificadores generados correctamente


,id_pais_cib,country,region,gci,nivel_madurez_gci
0,1,Afghanistan,Asia-Pacific,5.20,Iniciando (0-25)
1,2,Albania,Europe,64.32,Avanzado (50-75)
2,3,Algeria,Africa,33.95,En desarrollo (25-50)
3,4,Andorra,Europe,26.38,En desarrollo (25-50)
4,5,Angola,Africa,12.99,Iniciando (0-25)


---
## SECCIÓN 7: Validación de Resultados

Al finalizar la limpieza y transformación se verifica que los tres DataFrames
resultantes sean consistentes: sin nulos, sin duplicados, con tipos correctos
y con las columnas derivadas esperadas.

### 7.1 — Validación de `df_threats_sel`

In [33]:
print('═' * 60)
print('VALIDACIÓN FINAL — df_threats_sel')
print('═' * 60)

# Dimensiones finales
print(f'Dimensiones     : {df_threats_sel.shape}')

# Verificar que no haya duplicados
print(f'Duplicados      : {df_threats_sel.duplicated().sum()}')

# Verificar que no queden nulos
print(f'Nulos totales   : {df_threats_sel.isnull().sum().sum()}')
print()

# Tipos de datos finales de cada columna
print('── Tipos de datos ──')
print(df_threats_sel.dtypes)
print()

# Vista previa del DataFrame transformado
print('── Vista previa ──')
df_threats_sel.head()

════════════════════════════════════════════════════════════
VALIDACIÓN FINAL — df_threats_sel
════════════════════════════════════════════════════════════
Dimensiones     : (3000, 5)
Duplicados      : 135
Nulos totales   : 0

── Tipos de datos ──
country                                str
attack_type                            str
target_industry                        str
incident_resolution_time_in_hours    int64
velocidad_respuesta                    str
dtype: object

── Vista previa ──


,country,attack_type,target_industry,incident_resolution_time_in_hours,velocidad_respuesta
0,China,Phishing,Education,63,Normal (24-72h)
1,China,Ransomware,Retail,71,Normal (24-72h)
2,India,Man-In-The-Middle,It,20,Rápida (<24h)
3,Uk,Ransomware,Telecommunications,7,Rápida (<24h)
4,Germany,Man-In-The-Middle,It,68,Normal (24-72h)


### 7.2 — Validación de `df_cib_eda_sel`

In [35]:
print('═' * 60)
print('VALIDACIÓN FINAL — df_cib_eda_sel')
print('═' * 60)

# Dimensiones finales
print(f'Dimensiones     : {df_cib_eda_sel.shape}')

# Verificar que no haya duplicados
print(f'Duplicados      : {df_cib_eda_sel.duplicated().sum()}')

# Verificar nulos — deben ser 0 tras la imputación
print(f'Nulos totales   : {df_cib_eda_sel.isnull().sum().sum()}')
print()

# Nulos por columna para mayor detalle
print('── Nulos por columna ──')
print(df_cib_eda_sel.isnull().sum())
print()

# Verificar que el error ortográfico fue corregido
print('── Categorías de región ──')
print(df_cib_eda_sel['region'].unique())
print()

# Tipos de datos finales
print('── Tipos de datos ──')
print(df_cib_eda_sel.dtypes)
print()

# Vista previa del DataFrame transformado
print('── Vista previa ──')
df_cib_eda_sel.head()

════════════════════════════════════════════════════════════
VALIDACIÓN FINAL — df_cib_eda_sel
════════════════════════════════════════════════════════════
Dimensiones     : (192, 9)
Duplicados      : 0
Nulos totales   : 0

── Nulos por columna ──
id_pais_cib          0
country              0
region               0
gci                  0
ncsi                 0
cei                  0
ddl                  0
nivel_madurez_gci    0
indice_compuesto     0
dtype: int64

── Categorías de región ──
<StringArray>
['Asia-Pacific', 'Europe', 'Africa', 'North America', 'South America']
Length: 5, dtype: str

── Tipos de datos ──
id_pais_cib             int64
country                   str
region                    str
gci                   float64
ncsi                  float64
cei                   float64
ddl                   float64
nivel_madurez_gci    category
indice_compuesto      float64
dtype: object

── Vista previa ──


,id_pais_cib,country,region,gci,ncsi,cei,ddl,nivel_madurez_gci,indice_compuesto
0,1,Afghanistan,Asia-Pacific,5.20,11.69,1.000,19.50,Iniciando (0-25),7.80
1,2,Albania,Europe,64.32,62.34,0.566,48.74,Avanzado (50-75),63.53
2,3,Algeria,Africa,33.95,33.77,0.721,42.81,En desarrollo (25-50),33.88
3,4,Andorra,Europe,26.38,75.32,0.286,68.46,En desarrollo (25-50),45.96
4,5,Angola,Africa,12.99,9.09,0.707,22.69,Iniciando (0-25),11.43


### 7.3 — Validación de `df_usenet_sel`

In [36]:
print('═' * 60)
print('VALIDACIÓN FINAL — df_usenet_sel')
print('═' * 60)

# Dimensiones finales
print(f'Dimensiones     : {df_usenet_sel.shape}')

# Verificar que no haya duplicados
print(f'Duplicados      : {df_usenet_sel.duplicated().sum()}')

# Verificar que no queden nulos
print(f'Nulos totales   : {df_usenet_sel.isnull().sum().sum()}')
print()

# Verificar coherencia del porcentaje (debe estar entre 0 y 100)
print('── Rango de Percentage_Internet_Users ──')
print(f'  Mínimo : {df_usenet_sel["percentage_internet_users"].min()}')
print(f'  Máximo : {df_usenet_sel["percentage_internet_users"].max()}')
print()

# Tipos de datos finales
print('── Tipos de datos ──')
print(df_usenet_sel.dtypes)
print()

# Vista previa del DataFrame transformado
print('── Vista previa ──')
df_usenet_sel.head()

════════════════════════════════════════════════════════════
VALIDACIÓN FINAL — df_usenet_sel
════════════════════════════════════════════════════════════
Dimensiones     : (215, 8)
Duplicados      : 0
Nulos totales   : 0

── Rango de Percentage_Internet_Users ──
  Mínimo : 1.7
  Máximo : 100.0

── Tipos de datos ──
id_pais_net                     int64
country                           str
region                            str
internet_users                  int64
percentage_internet_users     float64
population_2021                 int64
categoria_penetracion        category
usuarios_por_millon_hab         int64
dtype: object

── Vista previa ──


,id_pais_net,country,region,internet_users,percentage_internet_users,population_2021,categoria_penetracion,usuarios_por_millon_hab
0,1,China,Asia,1102140000,77.3,1425893465,Alta (60-80%),772947
1,2,India,Asia,881250000,62.6,1407563842,Alta (60-80%),626082
2,3,United States,Americas,311300000,92.4,336997624,Muy Alta (>80%),923745
3,4,Indonesia,Asia,215626156,78.8,273753191,Alta (60-80%),787666
4,5,Pakistan,Asia,170000000,70.8,240000000,Alta (60-80%),708333


### 7.4 — Tabla resumen de validación general

In [37]:
# Construir tabla comparativa con el estado final de los tres DataFrames
resumen = pd.DataFrame({
    'Dataset': [
        'df_threats_sel',
        'df_cib_eda_sel',
        'df_usenet_sel'
    ],
    'Filas': [
        len(df_threats_sel),
        len(df_cib_eda_sel),
        len(df_usenet_sel)
    ],
    'Columnas_finales': [
        df_threats_sel.shape[1],
        df_cib_eda_sel.shape[1],
        df_usenet_sel.shape[1]
    ],
    'Nulos': [
        df_threats_sel.isnull().sum().sum(),
        df_cib_eda_sel.isnull().sum().sum(),
        df_usenet_sel.isnull().sum().sum()
    ],
    'Duplicados': [
        df_threats_sel.duplicated().sum(),
        df_cib_eda_sel.duplicated().sum(),
        df_usenet_sel.duplicated().sum()
    ],
    'Estado': ['✅ Listo', '✅ Listo', '✅ Listo']
})

print('RESUMEN DE VALIDACIÓN FINAL')
print('=' * 60)
print(resumen.to_string(index=False))

RESUMEN DE VALIDACIÓN FINAL
       Dataset  Filas  Columnas_finales  Nulos  Duplicados  Estado
df_threats_sel   3000                 5      0         135 ✅ Listo
df_cib_eda_sel    192                 9      0           0 ✅ Listo
 df_usenet_sel    215                 8      0           0 ✅ Listo


---
## REFLEXIÓN INDIVIDUAL FINAL
# Jeremy Moreno
Esta semana comprendí que la transformación de datos dentro de un proceso ETL va mucho más allá de limpiar columnas: es la base para convertir datos crudos en información útil para decisiones de seguridad. En mi trabajo, el mayor volumen de análisis lo dedico a revisar alertas del DLP, y hoy ese proceso es casi completamente manual.
El DLP genera cientos de alertas diarias sobre posibles fugas de información. El problema no es la falta de datos sino su calidad: alertas con campos vacíos, categorías escritas de forma inconsistente entre políticas, y niveles de severidad que no siguen un criterio uniforme. Aplicar funciones de normalización de texto y categorización con pd.cut() me permitiría estructurar ese volumen de alertas automáticamente, identificar patrones de fuga recurrentes por usuario, departamento o tipo de archivo, y priorizar los casos que realmente requieren investigación.
De forma similar, los logs de Entra ID acumulan miles de eventos de autenticación, cambios de permisos y accesos condicionales. Hoy los reviso con filtros básicos. Con un pipeline de transformación podría automatizar la detección de comportamientos anómalos: accesos fuera de horario, escaladas de privilegios no autorizadas, o patrones de inicio de sesión desde ubicaciones inusuales, todo documentado y reproducible para cuando el análisis deba presentarse internamente.
